In [2]:
# %pip install duckdb

   ---------------------------------------- 0.0/15.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/15.4 MB ? eta -:--:--
   -- ------------------------------------- 1.0/15.4 MB 8.7 MB/s eta 0:00:02
   -------- ------------------------------- 3.1/15.4 MB 9.7 MB/s eta 0:00:02
   -------------- ------------------------- 5.5/15.4 MB 10.5 MB/s eta 0:00:01
   ----------------------- ---------------- 9.2/15.4 MB 12.3 MB/s eta 0:00:01
   ------------------------------- -------- 12.1/15.4 MB 12.9 MB/s eta 0:00:01
   ---------------------------------------  15.2/15.4 MB 13.3 MB/s eta 0:00:01
   ---------------------------------------- 15.4/15.4 MB 12.7 MB/s  0:00:01
Note: you may need to restart the kernel to use updated packages.


In [3]:
import duckdb

print("DuckDB version:", duckdb.__version__)

DuckDB version: 2.0.0


In [5]:
from pathlib import Path

candidates = [
    Path.cwd() / "data" / "raw",
    Path.cwd().parent / "data" / "raw",
]

raw_dir = next(
    (path for path in candidates if path.is_dir()),
    None
)

processed_dir = raw_dir.parent / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

print("Processed data folder:", processed_dir)

Processed data folder: C:\Users\joaqu\OneDrive\Escritorio\CV\Postulaciones\XX. UNIDO\Personal Project\GitHub\seizure-signals\data\processed


In [6]:
from pathlib import Path
import duckdb

# Paths to the Parquet file and DuckDB database
parquet_path = processed_dir / "ids_2011_2024.parquet"
db_path = processed_dir / "seizure_signals.duckdb"

# Connect to the database (creates the file if it doesn't exist)
con = duckdb.connect(str(db_path))

# Create a staging table from the Parquet file
con.execute("""
    CREATE TABLE IF NOT EXISTS ids_staging AS
    SELECT *
    FROM read_parquet(?)
""", [str(parquet_path)])

print("Database created:", db_path)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Database created: C:\Users\joaqu\OneDrive\Escritorio\CV\Postulaciones\XX. UNIDO\Personal Project\GitHub\seizure-signals\data\processed\seizure_signals.duckdb


In [7]:
con.execute("""
    SELECT COUNT(*) AS total_records
    FROM ids_staging
""").df()

,total_records
0,2841288


In [8]:
con.execute("""
    SELECT
        source_year,
        COUNT(*) AS total_records
    FROM ids_staging
    GROUP BY source_year
    ORDER BY source_year
""").df()

,source_year,total_records
0,2011,41806
1,2012,62090
2,2013,62036
3,2014,265457
4,2015,238294
5,2016,307817
6,2017,232409
7,2018,220003
8,2019,119368
9,2020,151682


In [9]:
con.execute("""
    DESCRIBE ids_staging
""").df()

,column_name,column_type,null,key,default,extra
0,Seizure Date,TIMESTAMP_NS,YES,None,None,None
1,Country/Territory of Seizure,VARCHAR,YES,None,None,None
2,ISO3,VARCHAR,YES,None,None,None
3,Subregion,VARCHAR,YES,None,None,None
4,Region,VARCHAR,YES,None,None,None
5,Drug/Substance,VARCHAR,YES,None,None,None
6,Quantity Seized,DOUBLE,YES,None,None,None
7,Measurement Unit,VARCHAR,YES,None,None,None
8,City,VARCHAR,YES,None,None,None
9,Administrative Region,VARCHAR,YES,None,None,None


In [10]:
con.execute("""
    SELECT
        COUNT(*) AS total_records,
        COUNT(*) FILTER (
            WHERE "Quantity Seized" IS NULL
        ) AS missing_quantities,
        COUNT(*) FILTER (
            WHERE "Quantity Seized" <= 0
        ) AS nonpositive_quantities
    FROM ids_staging
""").df()

,total_records,missing_quantities,nonpositive_quantities
0,2841288,0,142


In [11]:
con.close()